# Managed vs External Table Examples

This notebook shows the conceptual difference between managed and external Delta tables in Databricks.

The flow covers:

- creating a managed Delta table
- creating an external Delta table
- comparing metadata and storage intent

In [ ]:
managed_table = "main.demo.customer_spend_managed"
external_table = "main.demo.customer_spend_external"
external_path = "/Volumes/main/demo/external/customer_spend_external"

In [ ]:
base_data = [
    (1, "Asha", 1200.0),
    (2, "Miguel", 950.0),
    (3, "Lina", 780.0)
]

base_df = spark.createDataFrame(base_data, ["customer_id", "customer_name", "spend"])

## Managed Delta table

A managed table lets Databricks decide the storage location based on the configured managed storage pattern.

In [ ]:
base_df.write.format("delta").mode("overwrite").saveAsTable(managed_table)
print(f"Created or replaced managed table {managed_table}")

In [ ]:
display(spark.table(managed_table).orderBy("customer_id"))

## External Delta table

An external table points explicitly to a storage location instead of using the default managed-table path.

In [ ]:
(
    base_df.write.format("delta")
    .mode("overwrite")
    .option("path", external_path)
    .saveAsTable(external_table)
)

print(f"Created or replaced external table {external_table} at {external_path}")

In [ ]:
display(spark.table(external_table).orderBy("customer_id"))

## Compare metadata

The important difference is not the rows themselves. The difference is how the table is registered and where the data path is controlled.

In [ ]:
managed_detail_df = spark.sql(f"DESCRIBE DETAIL {managed_table}")
external_detail_df = spark.sql(f"DESCRIBE DETAIL {external_table}")

display(managed_detail_df)
display(external_detail_df)

## Practical notes

- Managed vs external describes storage-path ownership and lifecycle.
- Both examples here are Delta tables because both are written in Delta format.
- External-table governance becomes more important when using Unity Catalog external locations and storage credentials.